# RappiPay Fraud Detection Lab## Construye un Cortex Agent con Cortex Code**Autor**: Juan Camilo Villarreal | **Duracion**: 45 minutos---### ObjetivoConstruir un agente de AI conversacional (Cortex Agent) que responda preguntas sobre fraude de RappiPay en lenguaje natural, usando datos sinteticos realistas.### Que vas a aprender1. Configurar un ambiente de datos de fraude en Snowflake2. Usar **Cortex Code** para explorar datos y generar SQL3. Crear una **Semantic View** para ensenarle a la AI tu modelo de datos4. Crear un **Cortex Agent** y habilitarlo en **Snowflake Intelligence**5. Hacer preguntas en espanol sobre fraude y obtener respuestas instantaneas### Prerequisitos- Cuenta Snowflake con ACCOUNTADMIN ([Solicita tu cuenta aqui](https://go.dataops.live/rappy-day/instructions))- Cortex Code CLI instalado (`npm install -g @snowflake-labs/cortex-code`)### ContextoEres un Data Engineer en RappiPay, el brazo fintech de Rappi. Tu equipo de fraude necesita una forma rapida de consultar alertas, patrones sospechosos y metricas sin escribir SQL manualmente. Vas a construir un asistente AI que responda sus preguntas.

---## Task 1: Setup del Ambiente (10 min)**Objetivo**: Crear la base de datos con datos sinteticos de fraude de RappiPay.### Paso 1: Ejecutar el setup completoEl siguiente SQL crea la base de datos `RAPPIPAY_DB` con 4 schemas, tablas de transacciones, usuarios, merchants y alertas de fraude, mas datos sinteticos realistas del mercado colombiano/mexicano.> **Nota**: Si ya ejecutaste `one_click_run.sql` previamente, puedes saltar al Paso 2.

In [ ]:
-- Task 1: Setup del ambienteUSE ROLE ACCOUNTADMIN;-- Verificar si la DB ya existeSHOW DATABASES LIKE 'RAPPIPAY_DB';

### Paso 2: Validar los datosEjecuta estas queries para confirmar que todo esta correctamente cargado:

In [ ]:
-- Validar row countsSELECT 'TRANSACTIONS' AS tabla, COUNT(*) AS filas FROM RAPPIPAY_DB.RAW.TRANSACTIONSUNION ALL SELECT 'USERS', COUNT(*) FROM RAPPIPAY_DB.RAW.USERSUNION ALL SELECT 'MERCHANTS', COUNT(*) FROM RAPPIPAY_DB.RAW.MERCHANTSUNION ALL SELECT 'FRAUD_ALERTS', COUNT(*) FROM RAPPIPAY_DB.RAW.FRAUD_ALERTS;

In [ ]:
-- Vista rapida de los datosSELECT alert_type, severity, status, COUNT(*) AS totalFROM RAPPIPAY_DB.RAW.FRAUD_ALERTSGROUP BY 1, 2, 3ORDER BY 4 DESCLIMIT 15;

### Checklist Task 1- [ ] RAPPIPAY_DB existe con schemas RAW, CURATED, ANALYTICS, APP- [ ] 12,000+ transacciones, 600 usuarios, 120 merchants, 250 alertas- [ ] Datos son realistas (ciudades colombianas/mexicanas, montos en COP)

---## Task 2: Explorar Datos con Cortex Code (10 min)**Objetivo**: Usar Cortex Code para entender los datos y generar analisis de fraude.### Paso 1: Analizar el modelo de datosAbre **Cortex Code** (CLI o Desktop) y copia este prompt:```Conectate a RAPPIPAY_DB y analiza las tablas en los schemas RAW, CURATED y ANALYTICS. Dame un resumen del modelo de datos incluyendo:1. Que tablas existen y cuantos registros tiene cada una2. Como se relacionan entre si (foreign keys implicitas)3. Que campos son mas relevantes para analisis de fraudeEjecuta las queries necesarias para responder.```### Paso 2: Descubrir patrones de fraude```Usando los datos de RAPPIPAY_DB, analiza los patrones de fraude:1. Cual es la tasa de fraude general (transacciones flagged/declined vs total)?2. Que tipos de fraude son mas comunes en las alertas?3. Hay correlacion entre el monto de la transaccion y ser marcada como fraude?4. Que merchants tienen la tasa mas alta de transacciones sospechosas?Muestra los resultados con queries SQL ejecutables.```### Paso 3: Generar un reporte ejecutivo```Genera un resumen ejecutivo de 5 bullets sobre el estado del fraude en RappiPaybasado en los datos de RAPPIPAY_DB. Incluye metricas concretas que un VP de Riesgoquerria ver: tasa de fraude, monto total en riesgo, alertas sin resolver, merchants de alto riesgo. Usa SQL para obtener los datos reales.```

In [ ]:
-- Ejecuta este analisis para comparar con lo que Cortex Code generoSELECT     COUNT(*) AS total_transacciones,    COUNT(CASE WHEN status IN ('flagged', 'declined') THEN 1 END) AS sospechosas,    ROUND(COUNT(CASE WHEN status IN ('flagged', 'declined') THEN 1 END) * 100.0 / COUNT(*), 2) AS tasa_fraude_pct,    ROUND(SUM(CASE WHEN status IN ('flagged', 'declined') THEN amount ELSE 0 END), 0) AS monto_en_riesgo_copFROM RAPPIPAY_DB.RAW.TRANSACTIONS;

### Checklist Task 2- [ ] Cortex Code respondio correctamente sobre el modelo de datos- [ ] Identificaste al menos 3 patrones de fraude en los datos- [ ] Tienes metricas concretas para tu reporte

---## Task 3: Crear Semantic View para Cortex Analyst (10 min)**Objetivo**: Crear una Semantic View que ensenará a Cortex Analyst como consultar tus datos de fraude.### Que es una Semantic View?Una Semantic View es un objeto SQL que define el significado de tus datos: que columnas son metricas, cuales son dimensiones, como se relacionan las tablas, y que sinonimos usar. Es lo que permite que la AI genere SQL correcto cuando le preguntas en lenguaje natural.### Paso 1: Generar la Semantic View con Cortex Code```Crea una Semantic View SQL para la base de datos RAPPIPAY_DB que permita hacer preguntas en lenguaje natural sobre fraude. La vista debe:1. Basarse en la tabla RAPPIPAY_DB.RAW.TRANSACTIONS unida con USERS y MERCHANTS2. Definir metricas: total_transacciones, monto_total, tasa_fraude, alertas_activas3. Definir dimensiones: ciudad, pais, tipo_transaccion, canal, merchant, severidad4. Agregar sinonimos en espanol (sin acentos): "fraude", "sospechoso", "ciudad", "monto"5. Incluir filtros por pais, tipo de fraude, severidad, canalUsa la sintaxis CREATE OR REPLACE SEMANTIC VIEW. Ejecuta el SQL.```### Paso 2: Alternativa - SQL directoSi prefieres ejecutar directamente, aqui esta el SQL:

In [ ]:
-- Task 3: Crear Semantic View-- NOTA: La sintaxis exacta puede variar segun la version de tu cuenta.-- Cortex Code generara la version correcta para tu ambiente.-- Primero verificamos que las vistas necesarias existenSELECT 'TRANSACTIONS_ENRICHED' AS tabla, COUNT(*) AS filas FROM RAPPIPAY_DB.CURATED.TRANSACTIONS_ENRICHEDUNION ALLSELECT 'FRAUD_METRICS_HOURLY', COUNT(*) FROM RAPPIPAY_DB.ANALYTICS.FRAUD_METRICS_HOURLY;

### Paso 3: Verificar la Semantic ViewDespues de crearla (via Cortex Code o SQL directo):

In [ ]:
-- Verificar semantic views creadasSHOW SEMANTIC VIEWS IN SCHEMA RAPPIPAY_DB.ANALYTICS;

### Checklist Task 3- [ ] Semantic View creada en RAPPIPAY_DB.ANALYTICS- [ ] Incluye metricas de fraude (tasa, monto, alertas)- [ ] Tiene sinonimos en espanol sin acentos- [ ] SHOW SEMANTIC VIEWS muestra la vista

---## Task 4: Crear Cortex Agent + Snowflake Intelligence (10 min)**Objetivo**: Crear un agente AI que responda preguntas sobre fraude usando tu Semantic View.### Paso 1: Crear el Cortex Agent con Cortex Code```Crea un Cortex Agent en Snowflake para analisis de fraude de RappiPay. El agente debe:1. Nombre: RAPPIPAY_FRAUD_ANALYST2. Ubicacion: RAPPIPAY_DB.APP3. Usar la semantic view que creamos en RAPPIPAY_DB.ANALYTICS4. Instrucciones: "Eres un analista de fraude de RappiPay. Responde preguntas sobre    transacciones sospechosas, alertas de fraude, y metricas de riesgo. Siempre incluye    numeros concretos y menciona el periodo de tiempo. Responde en espanol."5. Preguntas de ejemplo:   - "Cuantas alertas de fraude estan abiertas?"   - "Cual es la tasa de fraude por ciudad?"   - "Que merchants tienen mas transacciones sospechosas?"6. Usar seleccion automatica de modelo7. Otorgar USAGE a PUBLIC para que este disponible en IntelligenceEjecuta todo el SQL necesario incluyendo grants.```### Paso 2: SQL alternativo directo

In [ ]:
-- Task 4: Crear Cortex Agent-- Adapta la semantic view name segun lo que creaste en Task 3CREATE OR REPLACE CORTEX AGENT RAPPIPAY_DB.APP.RAPPIPAY_FRAUD_ANALYST  COMMENT = 'Agente de analisis de fraude para RappiPay - responde en espanol'  SYSTEM_PROMPT = 'Eres un analista de fraude de RappiPay. Responde preguntas sobre transacciones sospechosas, alertas de fraude, y metricas de riesgo. Siempre incluye numeros concretos y menciona el periodo de tiempo. Responde en espanol.'  SAMPLE_QUESTIONS = (    'Cuantas alertas de fraude estan abiertas?',    'Cual es la tasa de fraude por ciudad?',    'Que merchants tienen mas transacciones sospechosas?',    'Cual es el monto total en riesgo hoy?',    'Que tipo de fraude es mas comun?'  );

### Paso 3: Habilitar en Snowflake Intelligence

In [ ]:
-- Grants para que el agente sea accesibleGRANT USAGE ON DATABASE RAPPIPAY_DB TO ROLE PUBLIC;GRANT USAGE ON SCHEMA RAPPIPAY_DB.APP TO ROLE PUBLIC;GRANT USAGE ON CORTEX AGENT RAPPIPAY_DB.APP.RAPPIPAY_FRAUD_ANALYST TO ROLE PUBLIC;-- Verificar que el agente existeSHOW CORTEX AGENTS IN SCHEMA RAPPIPAY_DB.APP;

### Paso 4: Probar el agentePrueba estas preguntas directamente en Snowflake Intelligence (menu lateral izquierdo en Snowsight):1. **"Cuantas alertas de fraude estan abiertas?"**2. **"Cual es la tasa de fraude por ciudad?"**  3. **"Que merchants tienen mas transacciones sospechosas?"**4. **"Cual es el monto total en riesgo en los ultimos 30 dias?"**5. **"Comparar fraude entre Colombia y Mexico"**### Checklist Task 4- [ ] Cortex Agent RAPPIPAY_FRAUD_ANALYST creado- [ ] Grants otorgados (USAGE a PUBLIC)- [ ] SHOW CORTEX AGENTS muestra el agente- [ ] Puedes abrir Snowflake Intelligence y hacer preguntas- [ ] El agente responde en espanol con datos correctos

---## Task 5: Validacion Final + Cleanup (5 min)**Objetivo**: Confirmar que todo funciona end-to-end y documentar.### Validacion completa

In [ ]:
-- Validacion final: todo el stackSELECT 'Database' AS componente, 'RAPPIPAY_DB' AS nombre, 'OK' AS estadoUNION ALL SELECT 'Tablas RAW',     (SELECT COUNT(*)::VARCHAR FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'RAW') || ' tablas', 'OK'UNION ALL SELECT 'Dynamic Tables',     (SELECT COUNT(*)::VARCHAR FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'CURATED' AND TABLE_TYPE = 'BASE TABLE') || ' objetos', 'OK'UNION ALL SELECT 'Analytics',     (SELECT COUNT(*)::VARCHAR FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'ANALYTICS') || ' objetos', 'OK';

### Preguntas avanzadas para probarUsa estas preguntas para impresionar en tu presentacion:| Pregunta | Lo que demuestra ||----------|-----------------|| "Dame un resumen ejecutivo del fraude este mes" | Sintesis AI || "Que patron de fraude esta creciendo mas rapido?" | Tendencias || "Si soy un investigador nuevo, por donde empiezo?" | Contexto || "Compara el riesgo entre merchants de supermercado vs transporte" | Segmentacion |### Cleanup (cuando termines)Para eliminar todo lo creado:```sqlDROP DATABASE IF EXISTS RAPPIPAY_DB;DROP WAREHOUSE IF EXISTS RAPPIPAY_WH;DROP ROLE IF EXISTS RAPPIPAY_DE_ROLE;DROP ROLE IF EXISTS RAPPIPAY_ANALYST_ROLE;```---## Resumen de lo que construiste| Componente | Producto Snowflake | Para que sirve ||-----------|-------------------|----------------|| Datos sinteticos | Tables + Dynamic Tables | Pipeline de fraude Bronze > Silver > Gold || Semantic View | Cortex Analyst | Ensena a la AI tu modelo de datos || Cortex Agent | Snowflake Intelligence | Chat en lenguaje natural sobre fraude || Exploracion | Cortex Code | Generacion de SQL y analisis asistido por AI |**Siguiente paso**: Comparte el agente con tu equipo de fraude y pideles que hagan preguntas reales. Los patrones que descubran pueden alimentar nuevas reglas de deteccion.---*Lab creado por Juan Camilo Villarreal | Snowflake SE LATAM*